[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Tamaño del efecto.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo III: *Inferencia estadística*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se reportará el tamaño del efecto (d de Cohen, V de Cramér y η²) junto con el p-value y se integrarán las pruebas de hipótesis vistas en el módulo.

**Al terminar podrás:**

* elegir el tamaño del efecto que acompaña a cada prueba: d de Cohen, diferencia de proporciones, riesgo relativo, razón de momios, V de Cramér, η² o probabilidad de superioridad;
* leerlo en unidades originales, en desviaciones estándar y en frecuencias naturales;
* acompañarlo con su intervalo *bootstrap*;
* redactar un hallazgo con la plantilla, sin lenguaje causal.

## 🔁 Antes de empezar: repaso de memoria.

Sin abrir `21_proporciones_y_mas_de_dos_grupos.ipynb`, escribe con tus palabras, en tres o cuatro frases, la idea clave de ese *notebook*: cómo se comparan dos proporciones y dos variables de categorías, y qué se hace cuando hay tres o más grupos: qué dice la H1 del ANOVA y qué se hace después. No es un cuestionario ni hay una sola respuesta correcta: se trata de reconstruir la idea de memoria, porque recordar sin mirar ayuda a fijarla. Cuando termines, compáralo con el resumen que está al inicio de la versión con soluciones de este *notebook* y corrige lo que haga falta.

**✍️ Tu respuesta:**

_(escribe aquí)_

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencias.

py311 no cubre inferencia, así que la referencia de la API es la documentación oficial: [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html) (`ttest_ind`, `mannwhitneyu`, `permutation_test`, `chi2_contingency`, `f_oneway`, `levene`), [*statsmodels* · pruebas de proporciones](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportions_ztest.html), [*statsmodels* · `pairwise_tukeyhsd`](https://www.statsmodels.org/stable/generated/statsmodels.stats.multicomp.pairwise_tukeyhsd.html) y [*statsmodels* · `anova_oneway`](https://www.statsmodels.org/stable/generated/statsmodels.stats.oneway.anova_oneway.html).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas y las funciones de *SciPy* y *statsmodels* que se usarán, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.proportion import proportions_ztest

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará si no existe), `titanic.csv` y `tips.csv`, y creará `tip_pct`.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
propinas = pd.read_csv(os.path.join(DATA, "tips.csv")).assign(tip_pct=lambda d: 100 * d["tip"] / d["total_bill"])
pinguinos.shape, titanic.shape, propinas.shape

### Objetos de los *notebooks* anteriores.

Este *notebook* es independiente: la siguiente celda vuelve a crear lo que se necesita de los *notebooks* anteriores.

* La siguiente celda definirá `d_cohen` y `eta_cuadrada`, presentadas en `20_pruebas_para_dos_grupos.ipynb` y `21_proporciones_y_mas_de_dos_grupos.ipynb`.

In [ ]:
def d_cohen(a, b):
    a, b = np.asarray(a), np.asarray(b)
    varianza_combinada = ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2)
    return (a.mean() - b.mean()) / np.sqrt(varianza_combinada)


def eta_cuadrada(valores, grupos):
    datos = pd.DataFrame({"valor": valores, "grupo": grupos}).dropna()
    media_global = datos["valor"].mean()
    por_grupo = datos.groupby("grupo")["valor"].agg(["count", "mean"])
    suma_entre = (por_grupo["count"] * (por_grupo["mean"] - media_global) ** 2).sum()
    suma_total = ((datos["valor"] - media_global) ** 2).sum()
    return suma_entre / suma_total

---
## 1. Tamaño del efecto y ejercicios de este *notebook*.

| Situación | Prueba | Tamaño del efecto |
|---|---|---|
| Dos medias | Welch | d de Cohen (y la diferencia con su IC, en unidades originales) |
| Dos distribuciones asimétricas | Mann-Whitney o permutación de medianas | Diferencia de medianas con IC *bootstrap* |
| Dos proporciones | *z* de proporciones | Diferencia de proporciones o riesgo relativo |
| Dos categóricas | Chi cuadrada | V de Cramér |
| Tres o más medias | ANOVA de Welch de entrada (Levene solo describe la dispersión) + Tukey | η² |

**Plantilla para redactar:** "En estos datos, [grupo A] tuvo [medida] [mayor/menor] que [grupo B] ([valor A] contra [valor B]; diferencia de [valor y unidades], IC 95% [a, b]; [prueba], *p* [valor]; [tamaño del efecto: d de Cohen, V de Cramér o η²] = [valor], un efecto [magnitud])." Sin verbos causales ("provoca", "hace que") cuando los datos son observacionales.

**¿Por qué hace falta el tamaño del efecto?** Un p-value chico dice que una diferencia así sería rara si, en promedio, los grupos fueran iguales; no dice cuánto difieren ni si importa. Con muestras enormes basta una diferencia diminuta para obtener un p-value chico (en `21_proporciones_y_mas_de_dos_grupos.ipynb` se simula un caso con *p* ≈ 8×10⁻⁶ y d ≈ 0.02): se detecta, pero es demasiado pequeña para que importe en la práctica. El tamaño del efecto contesta la otra pregunta, «¿cuánto?», y se puede dar de **tres maneras**:
* en las **unidades originales**: los machos Adelie pesan unos 675 g más que las hembras;
* en **desviaciones estándar** (la d de Cohen): d = 2.17;
* en **frecuencias naturales**: en unas 94 de cada 100 parejas macho–hembra de Adelie, el macho pesa más.

**Las escalas son orientativas.** Los valores 0.2, 0.5 y 0.8 de la d son convenciones de aproximación: una d de 0.2 puede ser muy importante en salud pública y una de 0.8 irrelevante para otra decisión. Siempre se leen junto con la pregunta que se quiere contestar. Las escalas de V de Cramér (0.1, 0.3 y 0.5, solo para tablas con 2 filas o 2 columnas) y de η² (0.01, 0.06 y 0.14) están en `21_proporciones_y_mas_de_dos_grupos.ipynb`; V no es un porcentaje.

**Un contraste con datos reales.** Los Chinstrap y los Adelie difieren en 32 g de masa media (*p* = 0.59, d = 0.07: un efecto pequeño o nulo; el intervalo de la diferencia, de −86 a 150 g, no descarta diferencias de hasta unos 150 g), y los fumadores y no fumadores de `tips` dejan 16.3% y 15.9% de propina (*p* = 0.68, d = 0.06). En ambos casos no hay evidencia de un efecto, y se ven muy distintos de los 675 g y la d de 2.17 de los Adelie.

**¿Por qué esta sección?** El producto final de una prueba no es el p-value, sino una **frase** que una persona sin formación estadística pueda leer y que no exagere. La tabla resume qué tamaño del efecto acompaña a cada prueba, y la plantilla sirve para redactar los hallazgos de un informe (como en `25_estructura_del_informe_y_comunicacion.ipynb` y `26_del_notebook_al_informe.ipynb`).

**💡 Ideas clave de la sección 1:** cada prueba tiene su tamaño del efecto; se reporta magnitud, IC, prueba y p-value, en lenguaje descriptivo.

### Ejercicio 1a · Edad y supervivencia en el Titanic.

**🎯 Objetivo:** calcular una comparación de dos medias y redactarla con la plantilla. **Qué se espera aprender:** a comunicar magnitud, incertidumbre, prueba y tamaño del efecto en una sola frase, sin lenguaje causal, también cuando el p-value es chico pero el efecto es pequeño.

La masa de machos y hembras Adelie ya sirvió de ejemplo en la sección 1. Aquí se usará un caso nuevo: la **edad** (`age`, en años) de los pasajeros del Titanic que sobrevivieron y la de los que no sobrevivieron. 177 pasajeros no tienen edad registrada; quedan 290 que sobrevivieron y 424 que no, suficientes para que Welch funcione bien. Los datos son observados: describen a estos pasajeros, no una causa.

Sigue estos pasos con el *DataFrame* `titanic`:

1. **Hipótesis.** Escribe en palabras H0 y H1 (prueba bilateral).
2. **Prepara y describe.** Quita los faltantes de `age`, separa la edad de quienes sobrevivieron (`survived == 1`) y de quienes no (`survived == 0`) y calcula *n* y media de cada grupo.
3. **Welch e IC.** Aplica `stats.ttest_ind(..., equal_var=False)` y obtén el IC del 95% de la diferencia de medias (sobrevivieron menos no sobrevivieron). Con α = 0.05, decide si se **rechaza** o **no se rechaza** H0.
4. **Tamaño del efecto.** Calcula la d de Cohen con `d_cohen`.
5. **Redacción.** Con la plantilla de la sección 1, redacta la conclusión en una o dos frases. Di también si una diferencia de ese tamaño te parece importante y por qué.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### Ejercicio 1b · Aleta por especie.

**🎯 Objetivo:** recorrer el flujo completo para más de dos grupos (ANOVA de Welch → Tukey → η²), con Levene solo como descripción de la dispersión. **Qué se espera aprender:** a usar Welch por defecto, sin que Levene decida qué prueba usar, y a redactar la conclusión con el tamaño del efecto.

Compara `flipper_length_mm` entre las tres especies: calcula Levene solo para describir si la dispersión es parecida entre especies, aplica el ANOVA de Welch (y, para comparar, el ANOVA clásico), calcula η² y, si el ANOVA de Welch rechaza H0, Tukey. Redacta la conclusión.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### Ejercicio 1c · Puerto de embarque y supervivencia.

**🎯 Objetivo:** cuantificar una asociación entre categóricas. **Qué se espera aprender:** a acompañar la chi cuadrada con la V de Cramér y las tasas por grupo.

La clase del boleto y la supervivencia ya se analizaron en `21_proporciones_y_mas_de_dos_grupos.ipynb`. Aquí se usará otra variable del Titanic: el puerto donde embarcó cada pasajero (`embark_town`: Cherbourg, Queenstown o Southampton). Dos pasajeros no tienen puerto registrado; quedan 889, con al menos 77 por puerto.

Sigue estos pasos con el *DataFrame* `titanic`:

1. **Hipótesis.** Escribe en palabras H0 (el puerto y la supervivencia no están asociados) y H1.
2. **Tabla y tasas.** Arma la tabla con `pd.crosstab(titanic["embark_town"], titanic["survived"])` (`crosstab` deja fuera los faltantes) y calcula la tasa de supervivencia de cada puerto.
3. **Chi cuadrada.** Aplica `stats.chi2_contingency(tabla)` y reporta el estadístico, los grados de libertad, el p-value y la menor frecuencia esperada (debe ser de 5 o más).
4. **V de Cramér.** Calcúlala con `stats.contingency.association(tabla, method="cramer")`. La tabla tiene 2 columnas, así que aplican las referencias orientativas 0.1, 0.3 y 0.5.
5. **Conclusión.** Redáctala sin lenguaje causal. ¿Qué otras variables podrían venir junto con el puerto?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### Ejercicio 2 · Probabilidad de superioridad.

**🎯 Objetivo:** leer el tamaño del efecto en frecuencias naturales. **Qué se espera aprender:** a calcular la probabilidad de superioridad a partir de Mann-Whitney y a compararla con la d de Cohen.

La **probabilidad de superioridad** contesta: si se toma al azar un caso de cada grupo, ¿qué tan seguido el del primer grupo tiene el valor mayor? (los empates cuentan la mitad). Se obtiene al dividir el estadístico U de `stats.mannwhitneyu(a, b)` entre el número de parejas posibles, `len(a) * len(b)`. Un valor de 0.5 significa que ningún grupo tiende a ganar.

Las comparaciones de la sección 1 (machos y hembras Adelie, Chinstrap y Adelie, fumadores y no fumadores) ya están resueltas en las diapositivas. Aquí se usarán tres comparaciones nuevas, todas con al menos 34 casos por grupo:

1. la tarifa (`fare`) de primera clase contra la de segunda clase del Titanic (216 y 184 pasajeros);
2. la longitud del pico (`bill_length_mm`) de machos contra hembras Chinstrap (34 y 34);
3. `tip_pct` de mujeres contra hombres en `tips` (`sex` es el sexo de quien pagó; 87 y 157 mesas).

Para cada una, calcula la d de Cohen, la probabilidad de superioridad y las dos medianas. ¿Cómo se lee cada probabilidad en «parejas de cada 100»? ¿En cuál de las tres la d y la probabilidad de superioridad cuentan historias algo distintas, y por qué?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### Ejercicio 3 · Viajar solo y supervivencia en el Titanic.

**🎯 Objetivo:** comparar dos proporciones con varias medidas del tamaño del efecto. **Qué se espera aprender:** a distinguir la diferencia de proporciones, el riesgo relativo y la razón de momios, y a leer la V de Cramér con la escala que corresponde al tamaño de la tabla.

El sexo y la supervivencia ya se analizaron en `21_proporciones_y_mas_de_dos_grupos.ipynb` y en las diapositivas. Aquí se usará otra variable: `alone`, que vale `True` si el pasajero viajaba sin familiares a bordo (537 pasajeros) y `False` si viajaba acompañado (354). Con `pd.crosstab(titanic["alone"], titanic["survived"])` calcula:
* la tasa de supervivencia de quienes viajaban acompañados y de quienes viajaban solos, y su **diferencia en puntos** (acompañados menos solos);
* el **riesgo relativo** (tasa de los acompañados entre tasa de los solos);
* la **razón de momios** (los momios son sobrevivientes entre no sobrevivientes de cada grupo);
* la chi cuadrada sin corrección (`stats.chi2_contingency(tabla, correction=False)`) con su p-value, y la **V de Cramér** de viajar solo. Compárala con la del puerto del ejercicio 1c.

Redacta la conclusión sin lenguaje causal.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### Reto 1 · Diagnóstico gráfico de supuestos.

**🎯 Objetivo:** revisar los supuestos del ANOVA con gráficas. **Qué se espera aprender:** a juzgar si una desviación de los supuestos puede cambiar la conclusión.

Para el ejercicio 1b, dibuja un gráfico Q-Q de `flipper_length_mm` de cada especie (`stats.probplot(..., plot=ax)`) y un diagrama de caja por especie. ¿Las conclusiones del ANOVA dependen de algún supuesto dudoso?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

### Reto 2 · Intervalo *bootstrap* de la d de Cohen.

**🎯 Objetivo:** acompañar un tamaño del efecto con su incertidumbre. **Qué se espera aprender:** que la d también cambia de muestra a muestra y que su intervalo se puede obtener por *bootstrap*, como en `18_bootstrap_e_interpretacion_de_intervalos.ipynb`.

Con los arreglos de edades del ejercicio 1a (si usaste otros nombres, cámbialos), calcula el IC 95% de la d de Cohen con `stats.bootstrap((edad_sobrevivio, edad_no_sobrevivio), d_cohen, n_resamples=9999, method="percentile", random_state=401, vectorized=False)`. ¿Qué dice el intervalo? ¿El efecto podría ser prácticamente nulo? ¿Podría ser mediano?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## ✅ Cierre.

**En este *notebook*:**
* El tamaño del efecto contesta «¿cuánto?» y se puede dar en unidades originales, en desviaciones estándar (d de Cohen) o en frecuencias naturales (probabilidad de superioridad).
* Para categóricas: la diferencia de proporciones (en puntos), el riesgo relativo, la razón de momios y la V de Cramér, cuyas referencias dependen del tamaño de la tabla.
* Para tres o más grupos: η².
* El tamaño del efecto también tiene incertidumbre, que se puede estimar con un intervalo *bootstrap*.
* Las referencias de Cohen son orientativas, y con *n* enorme significativo no implica importante.

**Práctica (en Classroom):** completar los ejercicios 1 a 3, si es posible un reto, y aplicar al menos una prueba con su tamaño del efecto a otra comparación de `penguins`, `titanic` o `tips`.

**Siguiente *notebook*:** `23_eda_completo_con_mpg.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>